### [ELECT] Votação por Seção - Bronze Layer

Import libs and start spark context

In [ ]:
import os, sys, requests, zipfile, io, tempfile
from datetime import datetime

sys.path.append(os.path.join(os.getcwd(), '/home/jovyan/work/src/core'))

import spark_session
import pyspark.sql.functions as F

In [ ]:
spark = spark_session.build_spark()

Start variables

In [ ]:
target_schema = 'brz_elect'
target_table  = 'votacao_secao'

state = 'RS'
years = [2022, 2018, 2014]

base_url = 'https://cdn.tse.jus.br/estatistica/sead/odsele/votacao_secao'

Create schema and ingest each year

In [ ]:
spark_session.run_sql(f'CREATE SCHEMA IF NOT EXISTS {target_schema};')

In [ ]:
for year in years:
    url = f'{base_url}/votacao_secao_{year}_{state}.zip'
    print(f'Downloading {url}...')

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    with tempfile.TemporaryDirectory() as tmpdir:
        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            zf.extractall(tmpdir)

        csv_files = [f for f in os.listdir(tmpdir) if f.lower().endswith('.csv')]
        print(f'  Found files: {csv_files}')

        for csv_file in csv_files:
            csv_path = os.path.join(tmpdir, csv_file)

            df = spark.read \
                      .option('header', 'true') \
                      .option('sep', ';') \
                      .option('encoding', 'latin1') \
                      .csv(csv_path)

            df = df.withColumn('ingestion_datetime', F.lit(datetime.now())) \
                   .withColumn('ingestion_year',     F.lit(year)) \
                   .withColumn('ingestion_state',    F.lit(state)) \
                   .withColumn('ingestion_file',     F.lit(url))

            spark_session.write_table(df, target_schema, target_table, 'append')
            print(f'  Written {df.count()} rows for year {year}')

In [ ]:
spark.stop()